# Loan Portfolio & Credit Risk Analysis
**End-to-end Python analysis | Pandas | Matplotlib | Seaborn**

## Objective
Analyze a 50-record loan portfolio sample to summarize originations, outstanding balances, loan status, grade, purpose, income, DTI and observed default/NPL flags.

**Important limitation:** This is a small sample, not a verified production bank dataset. Results are descriptive and do not establish causation, predict creditworthiness, or support lending decisions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path('../data/cleaned/loan_portfolio_cleaned.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['Issue_Date'])
pd.set_option('display.max_columns', None)
print('Rows and columns:', df.shape)
df.head()

## 1. Data quality checks

In [ ]:
print('Duplicate Loan_ID count:', df['Loan_ID'].duplicated().sum())
display(df.isna().sum().to_frame('Missing values'))
display(df.describe(include='all').T)
assert df['Loan_ID'].nunique() == len(df), 'Duplicate loan IDs found'
assert df['Loan_Amount'].gt(0).all(), 'Non-positive loan amount found'
assert df['Default_Flag'].isin([0,1]).all()
assert df['NPL_Flag'].isin([0,1]).all()
print('Basic validation checks passed.')

## 2. Portfolio KPI summary

In [ ]:
kpis = pd.DataFrame({
    'KPI': ['Total loans','Total originated amount','Average loan amount',
            'Outstanding balance','Defaulted loans','Default rate',
            'NPL loans','NPL rate','Average interest rate','Average DTI'],
    'Value': [df['Loan_ID'].nunique(), df['Loan_Amount'].sum(),
              df['Loan_Amount'].mean(), df['Outstanding_Balance'].sum(),
              df['Default_Flag'].sum(), df['Default_Flag'].mean(),
              df['NPL_Flag'].sum(), df['NPL_Flag'].mean(),
              df['Interest_Rate'].mean(), df['DTI'].mean()]
})
display(kpis)

## 3. Yearly origination analysis

In [ ]:
yearly = df.groupby('Issue_Year').agg(
    loan_count=('Loan_ID','count'),
    total_originated=('Loan_Amount','sum'),
    outstanding=('Outstanding_Balance','sum'),
    defaults=('Default_Flag','sum')
).reset_index()
yearly['default_rate'] = yearly['defaults'] / yearly['loan_count']
display(yearly)
plt.figure(figsize=(8,4))
plt.bar(yearly['Issue_Year'].astype(str), yearly['total_originated'])
plt.title('Total Loan Originations by Year'); plt.xlabel('Issue Year'); plt.ylabel('USD')
plt.tight_layout(); plt.show()

## 4. Grade and default indicator analysis

In [ ]:
grade = df.groupby('Grade').agg(
    loan_count=('Loan_ID','count'),
    defaults=('Default_Flag','sum'),
    originated=('Loan_Amount','sum'),
    avg_interest_rate=('Interest_Rate','mean')
).reset_index()
grade['default_rate'] = grade['defaults'] / grade['loan_count']
display(grade)
plt.figure(figsize=(8,4))
plt.bar(grade['Grade'], grade['defaults'])
plt.title('Defaulted Loans by Grade'); plt.xlabel('Grade'); plt.ylabel('Number of loans')
plt.tight_layout(); plt.show()

## 5. Loan purpose and outstanding balances

In [ ]:
purpose = df.groupby('Purpose').agg(
    loan_count=('Loan_ID','count'),
    originated=('Loan_Amount','sum'),
    outstanding=('Outstanding_Balance','sum')
).sort_values('outstanding', ascending=True)
display(purpose)
plt.figure(figsize=(9,5))
plt.barh(purpose.index, purpose['outstanding'])
plt.title('Outstanding Balance by Loan Purpose'); plt.xlabel('USD')
plt.tight_layout(); plt.show()

## 6. Loan status distribution

In [ ]:
status = df.groupby('Loan_Status').size().sort_values(ascending=False)
display(status.to_frame('loan_count'))
plt.figure(figsize=(6,6))
plt.pie(status.values, labels=status.index, autopct='%1.1f%%', startangle=90)
plt.title('Loan Count by Status'); plt.tight_layout(); plt.show()

## 7. Income band and DTI analysis

In [ ]:
income = df.groupby('Income_Band').agg(
    loan_count=('Loan_ID','count'),
    avg_dti=('DTI','mean'),
    defaults=('Default_Flag','sum'),
    outstanding=('Outstanding_Balance','sum')
).reset_index()
income['default_rate'] = income['defaults'] / income['loan_count']
display(income)
plt.figure(figsize=(8,4))
plt.bar(income['Income_Band'], income['defaults'])
plt.title('Defaulted Loans by Income Band'); plt.xlabel('Income Band'); plt.ylabel('Number of loans')
plt.tight_layout(); plt.show()

## 8. State-level portfolio analysis

In [ ]:
state = df.groupby('State').agg(
    loan_count=('Loan_ID','count'),
    originated=('Loan_Amount','sum'),
    outstanding=('Outstanding_Balance','sum'),
    defaults=('Default_Flag','sum')
).reset_index().sort_values('outstanding', ascending=False)
display(state)
plt.figure(figsize=(8,4))
plt.bar(state['State'], state['outstanding'])
plt.title('Outstanding Balance by State'); plt.xlabel('State'); plt.ylabel('USD')
plt.tight_layout(); plt.show()

## 9. Loan term analysis

In [ ]:
term = df.groupby('Term_Months').agg(
    loan_count=('Loan_ID','count'),
    avg_interest_rate=('Interest_Rate','mean'),
    defaults=('Default_Flag','sum')
).reset_index()
term['default_rate'] = term['defaults'] / term['loan_count']
display(term)

## 10. Business findings and limitations
Write findings from the outputs above, quoting the exact sample counts and amounts. Do not say that grade, income, DTI, or geography causes defaults. Confirm the source definitions for `Default_Flag` and `NPL_Flag`. The 50-record sample is too small for robust credit-risk modeling and should not be used for real lending decisions.